# 01 — Treinar Tokenizer BPE para Tutor de Teoria Musical

Este notebook treina um tokenizer **BPE (Byte Pair Encoding)** usando vários arquivos `.txt` dentro da pasta `data/`, especializado em textos de teoria musical, incluindo marcadores de tutoria como:

```text
<pergunta>
<resposta>
<topico>
<conteudo>
<exercicio>
<resolucao>
<gabarito>
```

Estrutura esperada:

```text
projeto/
├── data/
│   ├── teoria_musical_base.txt
│   ├── intervalos.txt
│   ├── escalas.txt
│   ├── acordes.txt
│   ├── campo_harmonico.txt
│   ├── ritmo_compasso.txt
│   ├── exercicios_resolvidos.txt
│   └── perguntas_respostas_tutor.txt
└── tokenizer/
```

In [ ]:
# Instale apenas se necessário.
import sys
import subprocess

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

pip_install("tokenizers>=0.15.0")

In [ ]:
from pathlib import Path

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.normalizers import NFKC
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.trainers import BpeTrainer

## 1. Configurações

In [ ]:
DATA_DIR = Path("data")
TOKENIZER_DIR = Path("tokenizer")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

vocab_size = 8000
min_frequency = 2

special_tokens = [
    "<pad>",
    "<bos>",
    "<eos>",
    "<unk>",
    "<pergunta>",
    "<resposta>",
    "<topico>",
    "<conteudo>",
    "<exercicio>",
    "<resolucao>",
    "<gabarito>",
]

## 2. Buscar todos os arquivos `.txt`

O tokenizer será treinado usando todos os arquivos `.txt` encontrados em `data/`.

In [ ]:
files = sorted(str(p) for p in DATA_DIR.glob("*.txt"))

if not files:
    raise FileNotFoundError(
        "Nenhum arquivo .txt encontrado em ./data. "
        "Crie a pasta data/ e coloque os arquivos do corpus nela."
    )

print(f"Arquivos encontrados: {len(files)}")
for f in files:
    print("-", f)

## 3. Treinar o tokenizer BPE

`ByteLevel` é útil porque lida melhor com acentos, quebras de linha, cifras e símbolos como `#`, `b`, `<` e `>`.

In [ ]:
tokenizer = Tokenizer(BPE(unk_token="<unk>"))

# Normaliza caracteres Unicode equivalentes.
tokenizer.normalizer = NFKC()

# Tokenização em nível de bytes.
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    special_tokens=special_tokens,
    show_progress=True,
)

tokenizer.train(files=files, trainer=trainer)

tokenizer_path = TOKENIZER_DIR / "tokenizer.json"
tokenizer.save(str(tokenizer_path))

print("Tokenizer salvo em:", tokenizer_path)
print("Tamanho final do vocabulário:", tokenizer.get_vocab_size())

## 4. Verificar IDs dos tokens especiais

In [ ]:
for token in special_tokens:
    print(f"{token:12s} -> {tokenizer.token_to_id(token)}")

## 5. Teste de tokenização

O teste usa o mesmo formato que o transformer verá durante o treinamento.

In [ ]:
exemplo = '''<pergunta>
O que é uma escala maior?
<resposta>
Uma escala maior é uma sequência de notas organizada pelo padrão tom, tom, semitom, tom, tom, tom, semitom. Em C maior, temos C, D, E, F, G, A, B e C.
'''

encoded = tokenizer.encode(exemplo)
decoded = tokenizer.decode(encoded.ids)

print("Texto original:")
print(exemplo)

print("\nIDs:")
print(encoded.ids[:80])

print("\nTokens:")
print(encoded.tokens[:80])

print("\nTexto decodificado:")
print(decoded)

## 6. Funções utilitárias

Estas funções são úteis para testar como os textos serão enviados ao modelo.

In [ ]:
PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")

def encode_text(text: str, add_bos: bool = True, add_eos: bool = True) -> list[int]:
    ids = tokenizer.encode(text).ids
    if add_bos:
        ids = [BOS_ID] + ids
    if add_eos:
        ids = ids + [EOS_ID]
    return ids

def decode_ids(ids: list[int]) -> str:
    ids = [i for i in ids if i not in {PAD_ID, BOS_ID, EOS_ID}]
    return tokenizer.decode(ids)

teste = encode_text("<pergunta>O que é intervalo?<resposta>")
print(teste)
print(decode_ids(teste))